# ChatGPT code

In [ ]:
# 常见微分方程模型：人口、食饵-捕食者、传染病与其他应用。

# 运行：python ode_modeling_examples.py
# 会在当前目录写出 ode_models_overview.png。
# 依赖：numpy, scipy, matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


plt.style.use("seaborn-v0_8-whitegrid")


def solve(rhs, t, y0, args=()):
    """以指定时间网格求解常微分方程。"""
    return solve_ivp(rhs, (t[0], t[-1]), y0, t_eval=t, args=args,
                     rtol=1e-8, atol=1e-10).y


def logistic(t, y, r, K):
    # y'=r*y*(1-y/K)：资源有限时的人口增长
    return [r * y[0] * (1 - y[0] / K)]


def lotka_volterra(t, y, a, b, c, d):
    # R: 食饵，F: 捕食者；R'=aR-bRF，F'=dRF-cF
    R, F = y
    return [a * R - b * R * F, d * R * F - c * F]


def seir(t, y, beta, sigma, gamma, n):
    # S易感、E潜伏、I感染、R移除；人口总数 n 保持不变
    S, E, I, R = y
    force = beta * S * I / n
    return [-force, force - sigma * E, sigma * E - gamma * I, gamma * I]


def cstr(t, y, q_over_v, cin, k):
    # 连续搅拌反应器：流入-流出 + 一阶反应消耗
    C = y[0]
    return [q_over_v * (cin - C) - k * C]


def main():
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
    fig.suptitle("Differential-equation models in mathematical modeling", fontsize=17)

    # 1. 指数增长与逻辑斯蒂增长
    t = np.linspace(0, 40, 401)
    p0, r, K = 50, 0.12, 1000
    exponential = p0 * np.exp(r * t)
    population = solve(logistic, t, [p0], (r, K))[0]
    ax = axes[0, 0]
    ax.plot(t, exponential, label="exponential: P'=rP")
    ax.plot(t, population, linewidth=2.5, label="logistic: P'=rP(1-P/K)")
    ax.axhline(K, color="black", linestyle="--", label="carrying capacity K")
    ax.set(title="Population growth", xlabel="time (years)", ylabel="population")
    ax.legend(fontsize=8)

    # 2. 食饵—捕食者
    t = np.linspace(0, 80, 1601)
    prey, predator = solve(lotka_volterra, t, [40, 9], (0.55, 0.028, 0.35, 0.009))
    ax = axes[0, 1]
    ax.plot(t, prey, label="prey R")
    ax.plot(t, predator, label="predator F")
    ax.set(title="Lotka–Volterra time series", xlabel="time", ylabel="population")
    ax.legend()
    ax = axes[0, 2]
    ax.plot(prey, predator, color="tab:purple")
    ax.scatter(prey[0], predator[0], color="black", zorder=3, label="initial state")
    ax.set(title="Predator–prey phase portrait", xlabel="prey R", ylabel="predator F")
    ax.legend(fontsize=8)

    # 3. SEIR：一个新冠式呼吸道传染病示例（假设参数，并非真实拟合）
    # beta=0.42/day, 潜伏期 1/sigma=4 天, 传染期 1/gamma=7 天
    t = np.linspace(0, 180, 721)
    n = 1_000_000
    S, E, I, R = solve(seir, t, [n - 120, 80, 40, 0], (0.42, 1 / 4, 1 / 7, n))
    ax = axes[1, 0]
    ax.plot(t, S / 1000, label="S susceptible")
    ax.plot(t, E / 1000, label="E exposed")
    ax.plot(t, I / 1000, linewidth=2.5, label="I infectious")
    ax.plot(t, R / 1000, label="R removed/recovered")
    ax.set(title="SEIR epidemic (illustrative)", xlabel="days", ylabel="people (thousands)")
    ax.legend(fontsize=8, ncol=2)

    # 4. 干预比较：改变 beta（接触率）即可看到峰值差异
    ax = axes[1, 1]
    for beta, label in [(0.42, "baseline β=0.42"), (0.25, "intervention β=0.25")]:
        infected = solve(seir, t, [n - 120, 80, 40, 0], (beta, 1 / 4, 1 / 7, n))[2]
        ax.plot(t, infected / 1000, linewidth=2.2, label=label)
    ax.set(title="Contact reduction flattens the curve", xlabel="days", ylabel="infectious (thousands)")
    ax.legend(fontsize=8)

    # 5. 工程/环境：连续搅拌反应器浓度趋于平衡
    t = np.linspace(0, 20, 401)
    C = solve(cstr, t, [0], (0.8, 10, 0.35))[0]
    steady = 0.8 * 10 / (0.8 + 0.35)
    ax = axes[1, 2]
    ax.plot(t, C, linewidth=2.5, label="reactant concentration C")
    ax.axhline(steady, color="black", linestyle="--", label=f"steady state = {steady:.2f}")
    ax.set(title="CSTR / pollutant treatment", xlabel="residence-time units", ylabel="concentration")
    ax.legend(fontsize=8)

    fig.savefig("ode_models_overview.png", dpi=180, bbox_inches="tight")
    print("Saved: ode_models_overview.png")


if __name__ == "__main__":
    main()